In [8]:
import pandas as pd
from datetime import datetime, timedelta
import numpy as np
import os
import time

In [9]:
folder_paths = {
    "hc":r'C:\Users\huuchinh.nguyen\Downloads\HC ETG.xlsx',
}

hc_active = pd.read_excel(folder_paths['hc'])
hc_active.head(5)

,Joined Month,Account,Country,Function,Department,WD EID,Emp Name,Emp CNX Email,Supervisor Emp Code,Supervisor Name,...,Genesys Id,Actual Stage,Nesting Date,Tenured date,Employee Status,Is Tenured,Actual Stage 1,Date Changed,Start Unproductive,End Unproductive
0,2023-05-22,Etraveli,Vietnam,Ops,Operations,102254114,Truong Kim Hoang,kimhoang.truong@concentrix.com,102868033,Ung Thanh Phuong,...,7be34d71-9f11-4647-a478-4f76c1499bfe,Production,2023-11-17,2024-02-15,Active,Tenured,Production,NaT,NaN,NaN
1,2023-05-29,Etraveli,Vietnam,Ops,Operations,102257337,Vo Thi Hieu,thihieu.vo@concentrix.com,102474782,Ly Minh Hieu,...,4f9c2826-86b5-4ad5-9425-e2de7b477255,Production,2023-11-17,2024-02-15,Active,Tenured,Production,NaT,NaN,NaN
2,2023-06-26,Etraveli,Vietnam,Ops,Operations,102273439,Nguyen Duc Thinh,ducthinh.nguyen@concentrix.com,103225790,Diep Bao Ngan,...,a1e6b5df-26f0-4848-b2a7-ccfbad768ae8,Production,2023-11-17,2024-02-15,Active,Tenured,Production,NaT,NaN,NaN
3,2023-06-19,Etraveli,Vietnam,Ops,Operations,102269095,Do Thi Thao Vi,thithaovi.do@concentrix.com,103083135,Tran Huu Long,...,81c663dc-71f9-482d-a8d3-a926311a5d14,Production,2023-11-17,2024-02-15,Active,Tenured,Production,2025-07-28,NaN,NaN
4,2023-06-26,Etraveli,Vietnam,Ops,Operations,102273288,Allacnia Vu Ngoc Thy,vungocthy.allacnia@concentrix.com,102838562,Vu Minh Hung,...,a8fd0499-6ee8-4c93-a1c0-4330e7052f04,Production,2023-11-17,2024-02-15,Active,Tenured,Production,NaT,NaN,NaN


In [10]:
print(hc_active.columns)

Index(['Joined Month', 'Account', 'Country', 'Function', 'Department',
       'WD EID', 'Emp Name', 'Emp CNX Email', 'Supervisor Emp Code',
       'Supervisor Name', 'Supervisor Work Email', 'Manager Emp Code',
       'Manager Name', 'Batch Code/ Wave', 'AHA/B&M', 'Joining Date',
       'Training Start Date', 'Training End Date', 'Go Live Date',
       'Designation', 'Worker Category', 'LOB', 'Sub LOB', 'Site', 'Emp Type',
       'Gender', 'By Industry', 'By Channel', 'Process Service',
       'Pricing Model', 'Language', 'Owning Country', 'Last Working Date',
       'Termination Reason', 'Edvin user', 'Genesys Id', 'Actual Stage',
       'Nesting Date', 'Tenured date', 'Employee Status', 'Is Tenured',
       'Actual Stage 1', 'Date Changed', 'Start Unproductive',
       'End Unproductive'],
      dtype='object')


In [11]:
start_date = datetime(2022, 1, 1)
end_date = datetime.now() + timedelta(days=150)
date_list = pd.date_range(start_date, end_date)

date_table = pd.DataFrame(date_list, columns=['Date'])
expanded_hc = pd.DataFrame(
    [(date, wd_id) for wd_id in hc_active['WD EID'] for date in date_list],
    columns=['Date', 'WD EID']
)

# 7. Merge bảng expanded với dữ liệu gốc để gắn thông tin nhân viên lên từng ngày
HC_Data_Edition = expanded_hc.merge(hc_active, on='WD EID', how='left')

# 8. Danh sách các cột ngày tháng cần chuyển đổi định dạng datetime
date_columns = ['Joined Month', 'Joining Date','Training Start Date', 'Training End Date', 'Go Live Date','Nesting Date', 'Tenured date','Date Changed', 'Start Unproductive','End Unproductive']

# 9. Chuẩn hóa định dạng ngày tháng cho các trường ngày trong bảng
for column in date_columns:HC_Data_Edition[column] = pd.to_datetime(HC_Data_Edition[column], errors='coerce')

# 10. Xử lý dữ liệu thiếu (missing) và loại bỏ dòng trùng lặp, không hợp lệ
HC_Data_Edition.fillna(pd.NA, inplace=True)
HC_Data_Edition = HC_Data_Edition.sort_values(by=["WD EID", "Date"], ascending=[True, True])
HC_Data_Edition = HC_Data_Edition.drop_duplicates(subset=['Date', "WD EID"])
HC_Data_Edition = HC_Data_Edition[HC_Data_Edition["WD EID"].notnull()]

HC_Data_Edition

,Date,WD EID,Joined Month,Account,Country,Function,Department,Emp Name,Emp CNX Email,Supervisor Emp Code,...,Genesys Id,Actual Stage,Nesting Date,Tenured date,Employee Status,Is Tenured,Actual Stage 1,Date Changed,Start Unproductive,End Unproductive
774040,2022-01-01,101736942,2021-03-08,Etraveli,Vietnam,Ops,Operations,Nguyen Cao Thanh Truc,caothanhtruc.nguyen@concentrix.com,102173355,...,e81a8b48-c73c-4c53-902e-d7332946ba82,Production,2025-06-04,2025-09-09,Active,NH,Production,NaT,NaT,NaT
774041,2022-01-02,101736942,2021-03-08,Etraveli,Vietnam,Ops,Operations,Nguyen Cao Thanh Truc,caothanhtruc.nguyen@concentrix.com,102173355,...,e81a8b48-c73c-4c53-902e-d7332946ba82,Production,2025-06-04,2025-09-09,Active,NH,Production,NaT,NaT,NaT
774042,2022-01-03,101736942,2021-03-08,Etraveli,Vietnam,Ops,Operations,Nguyen Cao Thanh Truc,caothanhtruc.nguyen@concentrix.com,102173355,...,e81a8b48-c73c-4c53-902e-d7332946ba82,Production,2025-06-04,2025-09-09,Active,NH,Production,NaT,NaT,NaT
774043,2022-01-04,101736942,2021-03-08,Etraveli,Vietnam,Ops,Operations,Nguyen Cao Thanh Truc,caothanhtruc.nguyen@concentrix.com,102173355,...,e81a8b48-c73c-4c53-902e-d7332946ba82,Production,2025-06-04,2025-09-09,Active,NH,Production,NaT,NaT,NaT
774044,2022-01-05,101736942,2021-03-08,Etraveli,Vietnam,Ops,Operations,Nguyen Cao Thanh Truc,caothanhtruc.nguyen@concentrix.com,102173355,...,e81a8b48-c73c-4c53-902e-d7332946ba82,Production,2025-06-04,2025-09-09,Active,NH,Production,NaT,NaT,NaT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
491355,2026-01-15,103351597,2025-08-18,Etraveli,Vietnam,Training,Operations,Ly Trung,trung.ly@concentrix.com,101731725,...,<NA>,Training,2025-10-13,2026-01-15,Active,NH,Training,NaT,NaT,NaT
491356,2026-01-16,103351597,2025-08-18,Etraveli,Vietnam,Training,Operations,Ly Trung,trung.ly@concentrix.com,101731725,...,<NA>,Training,2025-10-13,2026-01-15,Active,NH,Training,NaT,NaT,NaT
491357,2026-01-17,103351597,2025-08-18,Etraveli,Vietnam,Training,Operations,Ly Trung,trung.ly@concentrix.com,101731725,...,<NA>,Training,2025-10-13,2026-01-15,Active,NH,Training,NaT,NaT,NaT
491358,2026-01-18,103351597,2025-08-18,Etraveli,Vietnam,Training,Operations,Ly Trung,trung.ly@concentrix.com,101731725,...,<NA>,Training,2025-10-13,2026-01-15,Active,NH,Training,NaT,NaT,NaT


In [12]:
def get_detail_status(row):
    d = row["Date"]
    nl_golive = row["Go Live Date"]
    nl_nesting = row["Nesting Date"]
    nl_training = row["Training Start Date"]
    unvail = row["Joining Date"]

    if pd.notna(nl_golive) and pd.notna(d) and d >= nl_golive: return "Go Live"
    if pd.notna(nl_nesting) and pd.notna(d) and d >= nl_nesting: return "Nesting"
    if pd.notna(nl_nesting) and pd.notna(d) and nl_nesting <= d: return "Nesting"
    if pd.notna(nl_training) and pd.notna(d) and d >= nl_training and pd.isna(nl_nesting): return "Training"
    if pd.notna(nl_training) and pd.notna(d) and nl_training <= d < nl_nesting: return "Training"
    if pd.notna(unvail) and pd.notna(d) and d < nl_training: return "Unavailable"

    return "-"

HC_Data_Edition["Detail Status"] = HC_Data_Edition.apply(get_detail_status, axis=1)
HC_Data_Edition

,Date,WD EID,Joined Month,Account,Country,Function,Department,Emp Name,Emp CNX Email,Supervisor Emp Code,...,Actual Stage,Nesting Date,Tenured date,Employee Status,Is Tenured,Actual Stage 1,Date Changed,Start Unproductive,End Unproductive,Detail Status
774040,2022-01-01,101736942,2021-03-08,Etraveli,Vietnam,Ops,Operations,Nguyen Cao Thanh Truc,caothanhtruc.nguyen@concentrix.com,102173355,...,Production,2025-06-04,2025-09-09,Active,NH,Production,NaT,NaT,NaT,Unavailable
774041,2022-01-02,101736942,2021-03-08,Etraveli,Vietnam,Ops,Operations,Nguyen Cao Thanh Truc,caothanhtruc.nguyen@concentrix.com,102173355,...,Production,2025-06-04,2025-09-09,Active,NH,Production,NaT,NaT,NaT,Unavailable
774042,2022-01-03,101736942,2021-03-08,Etraveli,Vietnam,Ops,Operations,Nguyen Cao Thanh Truc,caothanhtruc.nguyen@concentrix.com,102173355,...,Production,2025-06-04,2025-09-09,Active,NH,Production,NaT,NaT,NaT,Unavailable
774043,2022-01-04,101736942,2021-03-08,Etraveli,Vietnam,Ops,Operations,Nguyen Cao Thanh Truc,caothanhtruc.nguyen@concentrix.com,102173355,...,Production,2025-06-04,2025-09-09,Active,NH,Production,NaT,NaT,NaT,Unavailable
774044,2022-01-05,101736942,2021-03-08,Etraveli,Vietnam,Ops,Operations,Nguyen Cao Thanh Truc,caothanhtruc.nguyen@concentrix.com,102173355,...,Production,2025-06-04,2025-09-09,Active,NH,Production,NaT,NaT,NaT,Unavailable
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
491355,2026-01-15,103351597,2025-08-18,Etraveli,Vietnam,Training,Operations,Ly Trung,trung.ly@concentrix.com,101731725,...,Training,2025-10-13,2026-01-15,Active,NH,Training,NaT,NaT,NaT,Go Live
491356,2026-01-16,103351597,2025-08-18,Etraveli,Vietnam,Training,Operations,Ly Trung,trung.ly@concentrix.com,101731725,...,Training,2025-10-13,2026-01-15,Active,NH,Training,NaT,NaT,NaT,Go Live
491357,2026-01-17,103351597,2025-08-18,Etraveli,Vietnam,Training,Operations,Ly Trung,trung.ly@concentrix.com,101731725,...,Training,2025-10-13,2026-01-15,Active,NH,Training,NaT,NaT,NaT,Go Live
491358,2026-01-18,103351597,2025-08-18,Etraveli,Vietnam,Training,Operations,Ly Trung,trung.ly@concentrix.com,101731725,...,Training,2025-10-13,2026-01-15,Active,NH,Training,NaT,NaT,NaT,Go Live


In [13]:
def get_aon(row):
    if row["Detail Status"] == "Go Live":
        if pd.isna(row["Go Live Date"]) or row["Date"] < row["Go Live Date"]:
            return None
        else:
            return (row["Date"] - row["Go Live Date"]).days
    elif row["Detail Status"] == "Go Live Date":
        if pd.isna(row["Go Live Date"]) or row["Date"] < row["Go Live Date"]:
            return None
        else:
            return (row["Date"] - row["Go Live Date"]).days
    else:
        return None

HC_Data_Edition["AON"] = HC_Data_Edition.apply(get_aon, axis=1)

def get_tenure(row):
    if row["Detail Status"] in ["Nesting"]:
        return "Nesting"
    elif pd.notna(row["Go Live Date"]) and row["Date"] >= row["Go Live Date"]:
        aon = row["AON"]
        if pd.isna(aon) or aon == "-":
            return "-"
        elif 0 <= aon <= 30:
            return "1 to 30 Days"
        elif 30 < aon <= 60:
            return "31 to 60 Days"
        elif 60 < aon <= 90:
            return "61 to 90 Days"
        elif 90 < aon <= 120:
            return "91 to 120 Days"
        elif aon > 120:
            return "> 121 Days"
        
    return None

HC_Data_Edition["Tenure"] = HC_Data_Edition.apply(get_tenure, axis=1)


monday_of_week = HC_Data_Edition['Date'] - pd.to_timedelta(HC_Data_Edition['Date'].dt.weekday, unit='D')

HC_Data_Edition['Date_Start_Week'] = np.where(
    monday_of_week.notna(),
    'WB_' + monday_of_week.dt.strftime('%d/%m/%y'),
    pd.NA
)

HC_Data_Edition['Week_No'] = HC_Data_Edition['Date'].dt.isocalendar().week.astype('Int64')


HC_Data_Edition = HC_Data_Edition[['Week_No', 'Date_Start_Week', 'Date', 'WD EID', 'Emp Name', 'Emp CNX Email', 'Supervisor Name','Manager Name','Joining Date', 'Training Start Date', 'Training End Date', 'Go Live Date','Designation','LOB','Sub LOB', 'Site','Nesting Date', 'Detail Status', 'AON', 'Tenure']]
HC_Data_Edition

,Week_No,Date_Start_Week,Date,WD EID,Emp Name,Emp CNX Email,Supervisor Name,Manager Name,Joining Date,Training Start Date,Training End Date,Go Live Date,Designation,LOB,Sub LOB,Site,Nesting Date,Detail Status,AON,Tenure
774040,52,WB_27/12/21,2022-01-01,101736942,Nguyen Cao Thanh Truc,caothanhtruc.nguyen@concentrix.com,Le Hoang Nguyen,Tran Nguyen Tuong Vi,2021-03-08,2025-04-07,2025-06-10,2025-06-11,"Advisor II, Customer Service",English,English,QTSC,2025-06-04,Unavailable,NaN,None
774041,52,WB_27/12/21,2022-01-02,101736942,Nguyen Cao Thanh Truc,caothanhtruc.nguyen@concentrix.com,Le Hoang Nguyen,Tran Nguyen Tuong Vi,2021-03-08,2025-04-07,2025-06-10,2025-06-11,"Advisor II, Customer Service",English,English,QTSC,2025-06-04,Unavailable,NaN,None
774042,1,WB_03/01/22,2022-01-03,101736942,Nguyen Cao Thanh Truc,caothanhtruc.nguyen@concentrix.com,Le Hoang Nguyen,Tran Nguyen Tuong Vi,2021-03-08,2025-04-07,2025-06-10,2025-06-11,"Advisor II, Customer Service",English,English,QTSC,2025-06-04,Unavailable,NaN,None
774043,1,WB_03/01/22,2022-01-04,101736942,Nguyen Cao Thanh Truc,caothanhtruc.nguyen@concentrix.com,Le Hoang Nguyen,Tran Nguyen Tuong Vi,2021-03-08,2025-04-07,2025-06-10,2025-06-11,"Advisor II, Customer Service",English,English,QTSC,2025-06-04,Unavailable,NaN,None
774044,1,WB_03/01/22,2022-01-05,101736942,Nguyen Cao Thanh Truc,caothanhtruc.nguyen@concentrix.com,Le Hoang Nguyen,Tran Nguyen Tuong Vi,2021-03-08,2025-04-07,2025-06-10,2025-06-11,"Advisor II, Customer Service",English,English,QTSC,2025-06-04,Unavailable,NaN,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
491355,3,WB_12/01/26,2026-01-15,103351597,Ly Trung,trung.ly@concentrix.com,Nguyen Pham Thanh Phong,Hai Van Le,2025-08-18,2025-08-18,2025-10-16,2025-10-17,"Advisor I, Customer Service",English,English,QTSC,2025-10-13,Go Live,90.0,61 to 90 Days
491356,3,WB_12/01/26,2026-01-16,103351597,Ly Trung,trung.ly@concentrix.com,Nguyen Pham Thanh Phong,Hai Van Le,2025-08-18,2025-08-18,2025-10-16,2025-10-17,"Advisor I, Customer Service",English,English,QTSC,2025-10-13,Go Live,91.0,91 to 120 Days
491357,3,WB_12/01/26,2026-01-17,103351597,Ly Trung,trung.ly@concentrix.com,Nguyen Pham Thanh Phong,Hai Van Le,2025-08-18,2025-08-18,2025-10-16,2025-10-17,"Advisor I, Customer Service",English,English,QTSC,2025-10-13,Go Live,92.0,91 to 120 Days
491358,3,WB_12/01/26,2026-01-18,103351597,Ly Trung,trung.ly@concentrix.com,Nguyen Pham Thanh Phong,Hai Van Le,2025-08-18,2025-08-18,2025-10-16,2025-10-17,"Advisor I, Customer Service",English,English,QTSC,2025-10-13,Go Live,93.0,91 to 120 Days


In [14]:
HC_Data_Edition = HC_Data_Edition.to_excel("ETG tenure.xlsx", index=False)